In [1]:
# 答案空间：4 个岗位方向（第 5 个选项"都不合适"后面用 Noul 原语处理）
TRACKS = {
    "backend":  "后端开发：要求 Python/Go，熟悉数据库与 API 设计",
    "frontend": "前端开发：要求 JavaScript/TypeScript，熟悉 React 或 Vue",
    "product":  "产品经理：要求需求分析、原型设计、跨部门沟通经验",
    "data":     "数据分析：要求 SQL、Python 数据处理，有业务分析经验",
}

RESUME_1 = """
李小红，3 年工作经验：
- Python 后端开发，独立设计过两套 RESTful API
- MySQL 索引优化，把慢查询从 2s 降到 50ms
- 带过 2 人小组，负责过发版流程
"""

# state：只放"模型做判断需要的信息"，不放答案
state = {"resume": RESUME_1, "tracks": TRACKS}

import json
print(json.dumps(state, ensure_ascii=False, indent=2))

{
  "resume": "\n李小红，3 年工作经验：\n- Python 后端开发，独立设计过两套 RESTful API\n- MySQL 索引优化，把慢查询从 2s 降到 50ms\n- 带过 2 人小组，负责过发版流程\n",
  "tracks": {
    "backend": "后端开发：要求 Python/Go，熟悉数据库与 API 设计",
    "frontend": "前端开发：要求 JavaScript/TypeScript，熟悉 React 或 Vue",
    "product": "产品经理：要求需求分析、原型设计、跨部门沟通经验",
    "data": "数据分析：要求 SQL、Python 数据处理，有业务分析经验"
  }
}


In [2]:
# 判断调用入口（精简版）：有 Key 走真实模型，没 Key 走离线演示
from typesafe_sdk import Choice, Score, Noul
import os

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
client = None
if API_KEY:
    from typesafe_sdk import TypeSafeClient
    client = TypeSafeClient(api_key=API_KEY, model="jev-1.13.0")
    print("在线模式：调用真实模型")
else:
    print("离线模式：用手工答案演示代码路径（不调用真实模型）")

def ask(state, questions, offline_answers, label):
    if client is None:
        print(f"[offline] {label}：人工答案，仅演示代码路径")
        return offline_answers
    resp = client.system_one(state, questions)
    print(f"[live] {label}：真实模型返回")
    return resp.answers

离线模式：用手工答案演示代码路径（不调用真实模型）


In [3]:
questions = {
    "track": Choice(
        instructions="这份简历最适合哪个岗位方向？只看技能和经验匹配度。",
        criteria={"backend": "后端开发", "frontend": "前端开发",
                  "product": "产品经理", "data": "数据分析"}),
    "not_fit": Noul(instructions="这份简历是否与四个岗位方向都不匹配？"),
}

# 离线演示答案：手工构造，只练"怎么读返回、怎么做分支"
offline = {
    "track": {"type": "choice", "choice": "backend",
              "probabilities": {"backend": 0.78, "frontend": 0.06,
                                "product": 0.09, "data": 0.07},
              "confidence": 0.81},
    "not_fit": {"type": "noul", "noul": 0.08},
}

answers = ask(state, questions, offline, "简历初筛·第一次判断")
import json
print(json.dumps(answers, ensure_ascii=False, indent=2, default=str))

[offline] 简历初筛·第一次判断：人工答案，仅演示代码路径
{
  "track": {
    "type": "choice",
    "choice": "backend",
    "probabilities": {
      "backend": 0.78,
      "frontend": 0.06,
      "product": 0.09,
      "data": 0.07
    },
    "confidence": 0.81
  },
  "not_fit": {
    "type": "noul",
    "noul": 0.08
  }
}


In [4]:
# 决策函数：把"判断"变成"行动"。阈值是业务参数，调它不用重新训练模型。
def decide(answers, conf_threshold=0.7, noul_threshold=0.5):
    track = answers["track"]
    not_fit = answers["not_fit"]

    # 规则 1：Noul 说"都不合适" → 转人工（逃生口）
    if not_fit["noul"] >= noul_threshold:
        return {"action": "转人工",
                "reason": f"无匹配方向（noul={not_fit['noul']}）"}

    # 规则 2：置信度够高 → 采用模型的选择
    if track["confidence"] >= conf_threshold:
        return {"action": f"分配到：{track['choice']}",
                "reason": f"置信度 {track['confidence']} ≥ 阈值 {conf_threshold}"}

    # 规则 3：都不满足 → 人工复核
    return {"action": "转人工复核",
            "reason": f"置信度 {track['confidence']} < 阈值 {conf_threshold}"}

result = decide(answers)
import json
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "action": "分配到：backend",
  "reason": "置信度 0.81 ≥ 阈值 0.7"
}


In [5]:
# 换一份简历：厨师，跟四个技术岗都不沾边
RESUME_2 = """
王大厨，8 年工作经验：
- 粤菜主厨，擅长烧腊与点心
- 带过 12 人后厨团队，成本控制优秀
- 无任何 IT 或互联网从业经历
"""

state2 = {"resume": RESUME_2, "tracks": TRACKS}

offline2 = {
    "track": {"type": "choice", "choice": "backend",
              "probabilities": {"backend": 0.28, "frontend": 0.22,
                                "product": 0.25, "data": 0.25},
              "confidence": 0.35},
    "not_fit": {"type": "noul", "noul": 0.85},
}

answers2 = ask(state2, questions, offline2, "简历初筛·压力测试")
result2 = decide(answers2)
import json
print(json.dumps(result2, ensure_ascii=False, indent=2))

[offline] 简历初筛·压力测试：人工答案，仅演示代码路径
{
  "action": "转人工",
  "reason": "无匹配方向（noul=0.85）"
}
